# Volatility Modeling in Markovian and Rough Regimes
## Signature Methods and Analytical Expansions

**ArXivist-generated reproduction notebook**  
Paper: [arXiv:2507.23392](https://arxiv.org/abs/2507.23392)  
Authors: Elisa Alòs, Òscar Burés, Rafael de Santiago, Josep Vives  
Generated: 2026-05-13

This notebook walks through the key components of the `volsig` implementation,
runs a small-scale calibration, and verifies the setup against the paper's
reported behavior on a mini experiment.

In [ ]:
# Cell 1: Environment Check
import sys
print(f'Python: {sys.version}')

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('Running on CPU — notebook demo will be fast (nMC is small)')
except ImportError:
    print('PyTorch not found — running NumPy-only mode')

import numpy as np
import scipy
print(f'NumPy: {np.__version__}')
print(f'SciPy: {scipy.__version__}')

In [ ]:
# Cell 2: Install the project (run once)
import subprocess, sys
from pathlib import Path

repo_root = Path('..').resolve()  # notebooks/ -> repo root
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(repo_root), '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('volsig installed successfully')
else:
    print('Installation warning (may already be installed):')
    print(result.stderr[:500])

## Paper Overview

### Problem
Calibrating a stochastic volatility model to market-observed implied volatility (IV) surfaces
is a core task in derivatives trading. Given option prices at various strikes and maturities,
we need to find model parameters that reproduce those prices.

### Two Approaches

**Approach A — Analytical expansions** (Section 2):  
For the *Heston model*, a second-order asymptotic expansion of the IV surface in three
regimes (short-T, ATM, long-T) gives a 3-equation calibration system solvable in closed form.
For the *rough Bergomi model*, a new 4-step VIX-based procedure estimates H, η, ρ, σ₀
sequentially from short-maturity data.

**Approach B — Signature-based model** (Section 4):  
Volatility is modelled as a *linear functional of the truncated path signature*:
$$\sigma_t(\ell) = \sum_{|I| \leq N} \ell_I \langle e_I,\, S(\hat{X})^{\leq N}_t \rangle$$
where $\hat{X}_t = (t, X_t)$ is the time-augmented primary process and $S(\hat{X})^{\leq N}$
is its truncated signature at level $N=3$. The discounted price process (Proposition 4.2) is:
$$\tilde{S}_T(\ell) = S_0 \exp\!\left(-\|U(T)\ell\|^2 + \ell^\top \int_0^T \mathrm{vec}(S^{\leq N}_s)\,dZ_s\right)$$
Calibration minimises $L(\ell) = \sum_i \gamma_i (C^{\mathrm{mkt}}_i - C(K_i,T_i,\ell))^2$
via L-BFGS-B, where $\gamma_i = 1/\mathrm{Vega}_i$.

### Code map
| Paper Section | Code module |
|---|---|
| Sec 2.1 — Heston ASV | `models/heston.py` |
| Sec 2.2 — VIX calibration | `models/rough_bergomi.py` |
| Sec 3 — Signatures | `signatures/compute.py` |
| Sec 4 — Sig-vol model | `models/signature_vol.py` |
| Prop 4.2 — MC pricer | `pricing/mc_pricer.py` |
| Sec 4.2 — Calibration | `calibration/optimizer.py` |

## Component 1: Black-Scholes Utilities (`pricing/black_scholes.py`)

Used for implied volatility inversion and computing inverse-Vega weights $\gamma_i = 1/\mathrm{Vega}_i$.

In [ ]:
import sys; sys.path.insert(0, str(Path('..') / 'src'))
from volsig.pricing.black_scholes import BlackScholes

S0, K, T, r, sigma = 100.0, 100.0, 0.5, 0.0, 0.2
price = BlackScholes.call_price(S0, K, T, r, sigma)
iv    = BlackScholes.implied_vol(price, S0, K, T, r)
vega  = BlackScholes.vega(S0, K, T, r, sigma)

print(f'ATM Call price:  {price:.6f}')
print(f'Implied vol:     {iv:.6f}  (should recover σ={sigma})')
print(f'Vega:            {vega:.6f}')
print(f'Inverse-Vega γ:  {1/vega:.6f}')
assert abs(iv - sigma) < 1e-6, f'IV inversion error: {abs(iv-sigma):.2e}'
print('✓ BS round-trip passed')

## Component 2: Primary Process Simulation (`models/primary_process.py`)

The primary process $X_t$ drives the volatility signature. Two variants are used:

**Heston variance SDE** (Section 5):
$$dX_t = \kappa(\theta - X_t)\,dt + \nu\sqrt{X_t}\,dW_t$$

**Shifted-exp fBM** (Section 6, best variant):
$$X_t = X_0 \cdot \exp\!\left(\sqrt{2H}\int_0^t (t-s)^{H-1/2}\,dW_s\right)$$

In [ ]:
import numpy as np
from volsig.models.primary_process import PrimaryProcessSimulator

rng = np.random.default_rng(42)
nMC, T_steps, dt = 500, 50, 0.1/50  # mini demo
W = rng.standard_normal((nMC, T_steps)) * np.sqrt(dt)

# Heston variance
X_heston = PrimaryProcessSimulator.simulate(
    'heston_variance', W=W, dt=dt, nMC=nMC, T_steps=T_steps,
    X0=0.1, nu=0.2, kappa=2.0, theta=0.15
)
print(f'Heston variance:  shape={X_heston.shape}, mean(X_T)={X_heston[:,-1].mean():.4f}, min={X_heston.min():.4f}')

# Shifted-exp fBM
X_fbm = PrimaryProcessSimulator.simulate(
    'fbm_shifted_exp', W=W, dt=dt, nMC=nMC, T_steps=T_steps,
    X0=0.1, H=0.2
)
print(f'Shifted-exp fBM:  shape={X_fbm.shape}, mean(X_T)={X_fbm[:,-1].mean():.4f}, min={X_fbm.min():.4f}')
assert X_fbm.min() > 0, 'fBM shifted-exp should be strictly positive'
print('✓ Primary process simulation passed')

## Component 3: Signatures (`signatures/compute.py`)

### Time Augmentation (Section 3.4)
$$\hat{X}_t = (t,\, X_t) \in \mathbb{R} \oplus V$$

### Truncated Signature (Definition 3.7, Section 4.1)
For a 2D time-augmented path truncated at level $N=3$, the signature has
$\sum_{k=0}^{3} 2^k = 15$ coordinates:
$$S(\hat{X})^{\leq 3}_t = \left(1,\; t,\; X_t,\; \tfrac{t^2}{2},\; \int_0^t s\,dX_s,\; \int_0^t X_s\,ds,\; \int_0^t X_s\,dX_s,\; \ldots\right)$$

### Shuffle Product and Q-Matrix (Eq. 4.6)
$$Q(T)_{L(I),L(J)} = -\tfrac{1}{2}\langle (e_I \shuffle e_J) \otimes e_0,\; S(\hat{X})^{\leq 7}_T \rangle$$

In [ ]:
from volsig.signatures.compute import SignatureComputer, QMatrixAssembler, sig_dimension, shuffle_product

# Signature dimensions
for N in [1, 2, 3, 7]:
    print(f'  N={N}: sig_dimension(d=2, N={N}) = {sig_dimension(2, N)}')

# Shuffle product example (Section 3.1, Example 3.4)
sp = shuffle_product((1, 2), (3,))
print(f'\nShuffle (1,2) ⊔ (3,) = {dict(sp)}')
print(f'  (paper Example 3.4: should yield 3 terms with coefficient 1 each)')

In [ ]:
# Compute truncated signatures on mini paths
sc = SignatureComputer(d=2, N=3)

X_aug = sc.time_augment(X_heston, dt)
print(f'Time-augmented path shape: {X_aug.shape}  (nMC, T+1, 2)')

sig_paths = sc.compute_signature_paths(X_aug)
print(f'Signature paths shape:     {sig_paths.shape}  (nMC, T+1, 15)')

# Level-0 coefficient should always be 1
assert np.allclose(sig_paths[:, :, 0], 1.0), 'Level-0 signature must be 1'
# Level-1 time coordinate should equal cumulative time: sig^{(0)} = t
time_sig = sig_paths[:, :, 1]  # index 1 = multi-index (0,) = time integral
time_grid = np.arange(T_steps+1) * dt
assert np.allclose(time_sig[0], time_grid, atol=1e-10), 'Time signature mismatch'

print(f'✓ Signature level-0 = 1 everywhere')
print(f'✓ Signature level-1 time channel = cumulative time')
print(f'  Sample sig at T: {sig_paths[0, -1, :]}')

## Component 4: Q-Matrix and Cholesky (`signatures/compute.py`)

The Q-matrix is assembled from the **extended signature** (level 7, 255 dims)
using precomputed shuffle products. It must be negative semi-definite.

In [ ]:
# Q-matrix assembly (mini demo with 50 paths)
sc_ext = SignatureComputer(d=2, N=7)  # extended for Q
q_asm = QMatrixAssembler(N=3, d=2)

X_aug_mini = X_aug[:50]  # 50 paths
sig_ext = sc_ext.compute_terminal_signature(X_aug_mini)  # [50, 255]
print(f'Extended signature shape: {sig_ext.shape}  (50 paths, 255 dims)')

Q = q_asm.assemble(sig_ext)   # [50, 15, 15]
print(f'Q matrix shape: {Q.shape}')

# Check negative semi-definiteness of Q (average over paths)
Q_mean = Q.mean(axis=0)
eigvals = np.linalg.eigvalsh(Q_mean)
print(f'Q eigenvalues (should all be ≤ 0): min={eigvals.min():.4e}, max={eigvals.max():.4e}')
assert eigvals.max() < 1e-6, f'Q is not negative semi-definite: max eigenvalue = {eigvals.max()}'

U = q_asm.cholesky(Q)   # [50, 15, 15]
print(f'Cholesky factor shape: {U.shape}')
print(f'✓ Q is negative semi-definite')
print(f'✓ Cholesky factorisation succeeded')

## Component 5: MC Pricer (`pricing/mc_pricer.py`)

Given $U(T)$ and $\int_0^T \mathrm{vec}(S^{\leq N}_s)\,dZ_s$,
pricing for any $\ell$ is a single matrix-vector multiply + exp (Proposition 4.2):

$$\tilde{S}_T(\ell)(\omega_j) = S_0 \cdot \exp\!\Big(-\|U(T)(\omega_j)\,\ell\|^2 + \ell^\top \int \mathrm{vec}(S)\,dZ(\omega_j)\Big)$$

In [ ]:
from volsig.pricing.mc_pricer import SignatureMCPricer, compute_stochastic_integrals

# Stochastic integral
dZ = W[:50]  # [50, T_steps] (reusing W as a stand-in for dZ)
stoch_int = compute_stochastic_integrals(sig_paths[:50], dZ)  # [50, 15]
print(f'Stochastic integral shape: {stoch_int.shape}  (50 paths, 15 coords)')

# Pricer
T_demo = T_steps * dt
pricer = SignatureMCPricer(U=U, stoch_int=stoch_int, S0=100.0, r=0.0, maturity=T_demo)

# Price with paper's ℓ* for heston_uncorr (Section 5.1)
l_star_paper = np.array([
    0.201202133, 0.142660997, 1.08471290, -0.297312378, -0.0293435325,
    -0.0422317187, 9.25090162e-4, 0.293103687, -0.0143435573, -0.0134285652,
    -1.64737083e-3, -2.89883092e-3, -5.72798006e-4, -1.93045420e-3, -1.84406803e-4
])

S_T = pricer.simulate_terminal_prices(l_star_paper)
print(f'\nTerminal prices S_T: shape={S_T.shape}')
print(f'  mean={S_T.mean():.4f}, std={S_T.std():.4f}, min={S_T.min():.4f}')

call_price = pricer.price_call(l_star_paper, K=100.0)
print(f'\nCall price (K=100, T={T_demo:.3f}): {call_price:.6f}')
print(f'(Demo with 50 MC paths — expect variance; use nMC=800k for paper accuracy)')

## Component 6: Heston ASV Analytical Expansion (`models/heston.py`)

Three asymptotic expansions from Section 2.1 (Alòs et al. 2015):

**ATM** (Eq. 2.6): $I(T, K^*) \approx \sigma_0 + \frac{3\sigma_0^2\rho\nu - 6\kappa(\sigma_0^2-\theta) - \nu^2}{24\sigma_0}T$

**Short-T** (Eq. 2.4): $I(0, K) \approx \sigma_0 - \frac{\rho\nu}{4\sigma_0}(x-k) + \frac{\nu^2}{24\sigma_0^3}(x-k)^2$

**Long-T** (Eq. 2.5): $I(T, K^*) \approx \sqrt{\theta}\left(1 + \frac{\nu\rho}{4\kappa} - \frac{\nu^2}{32\kappa^2}\right) + \mathcal{O}(1/T)$

In [ ]:
from volsig.models.heston import HestonModel

# Paper parameters (Table 2.1, Section 2.1)
heston = HestonModel(sigma0=0.2, nu=0.3, kappa=3.0, theta=0.09, rho=0.0, S0=100.0, r=0.0)

strikes    = np.array([90., 95., 100., 105., 110.])
maturities = np.array([0.1, 0.6, 1.1, 1.6])

iv_asv = heston.implied_vol_surface_ASV(strikes, maturities)

print('ASV Implied Volatility Surface (Section 2.1 benchmark):')
print(f'        ' + '  '.join(f'K={int(k)}' for k in strikes))
for i, T in enumerate(maturities):
    row = '  '.join(f'{iv_asv[i,j]:.4f}' for j in range(len(strikes)))
    print(f'T={T:.1f}:  {row}')

# Verify ATM expansion
iv_atm_T01 = heston.iv_atm_approx(T=0.1)
print(f'\nATM IV at T=0.1 (Eq. 2.6): {iv_atm_T01:.6f}  (paper: ≈ σ₀=0.2)')

## Mini Calibration Demo (Section 4.3 Algorithm)

We run the full signature calibration pipeline end-to-end with:
- `nMC = 2,000` paths (vs paper's 800,000)
- `T_steps = 20` (vs paper's daily)
- `max_iter = 30` optimizer steps

This gives a quick sanity check that the pipeline runs without errors.
For paper-accurate results, use the full config.

In [ ]:
# Mini config — faster than paper's full config
from volsig.utils.config import Config, seed_everything
import io, yaml

mini_yaml = '''
model:
  signature_truncation_N: 3
  primary_process: heston_variance
  S0: 100.0
  r: 0.0
heston_primary:
  X0: 0.1
  x0_is_variance: true
  nu: 0.2
  kappa: 2.0
  theta: 0.15
  rho_asset_vol: 0.0
heston_market:
  sigma0: 0.2
  nu: 0.3
  kappa: 3.0
  theta: 0.09
  rho: 0.0
rbergomi_market:
  sigma0: 0.2
  H: 0.1
  eta: 0.5
  rho: -0.7
fbm_primary:
  H: 0.2
  X0: 0.1
  rho_asset_vol: -0.6
simulation:
  nMC: 2000
  T_steps_per_unit: 20
  seed: 42
  interpolation: linear
  fbm_method: cholesky
  cholesky_reg_eps: 1.0e-8
calibration:
  optimizer: L-BFGS-B
  tolerance: 1.0e-6
  l0_init: zeros
  box_bounds: [-10.0, 10.0]
  maturities: [0.1, 0.6]
  strikes: [90, 95, 100, 105, 110]
  weight_scheme: inverse_vega
  max_iter: 30
vix_calibration:
  Delta_trading_days: 30
  T1: 0.1
  T2: 0.5
hardware:
  device: cpu
  num_workers: 1
  use_float64: true
paths:
  output_dir: ../results/
  checkpoint_dir: ../results/checkpoints/
  data_dir: ../data/
'''

cfg = Config(**{
    k: v for k, v in yaml.safe_load(mini_yaml).items()
    if k in Config.__dataclass_fields__
})

from volsig.utils.config import (
    ModelConfig, HestonPrimaryConfig, FBMPrimaryConfig,
    HestonMarketConfig, RBergomiMarketConfig, SimulationConfig,
    CalibrationConfig, VIXCalibrationConfig, HardwareConfig, PathsConfig
)
raw = yaml.safe_load(mini_yaml)
cfg = Config(
    model=ModelConfig(**raw['model']),
    heston_primary=HestonPrimaryConfig(**raw['heston_primary']),
    fbm_primary=FBMPrimaryConfig(**raw['fbm_primary']),
    heston_market=HestonMarketConfig(**raw['heston_market']),
    rbergomi_market=RBergomiMarketConfig(**raw['rbergomi_market']),
    simulation=SimulationConfig(**raw['simulation']),
    calibration=CalibrationConfig(**raw['calibration']),
    vix_calibration=VIXCalibrationConfig(**raw['vix_calibration']),
    hardware=HardwareConfig(**raw['hardware']),
    paths=PathsConfig(**raw['paths']),
)
seed_everything(cfg.simulation.seed)
print(f'Mini config loaded: {cfg}')
print(f'  nMC={cfg.simulation.nMC}, N={cfg.model.signature_truncation_N}, maturities={cfg.calibration.maturities}')

In [ ]:
# Build multi-maturity pricer (offline precompute)
from volsig.models.signature_vol import SignatureVolModel

print('Building SignatureVolModel and precomputing signatures...')
sig_model = SignatureVolModel(cfg)
print(sig_model)

try:
    multi_pricer = sig_model.build_multi_maturity_pricer(
        maturities=cfg.calibration.maturities,
        seed=cfg.simulation.seed,
    )
    print(f'\n✓ MultiMaturityPricer built: {multi_pricer}')
except Exception as e:
    print(f'ERROR during precompute: {e}')
    raise

In [ ]:
# Generate synthetic market prices
from volsig.pricing.black_scholes import BlackScholes

strikes    = np.array(cfg.calibration.strikes)
maturities_arr = np.array(cfg.calibration.maturities)

# Use ASV approximation as proxy for market prices (fast, no MC needed for demo)
heston_mkt = HestonModel(
    sigma0=cfg.heston_market.sigma0, nu=cfg.heston_market.nu,
    kappa=cfg.heston_market.kappa, theta=cfg.heston_market.theta,
    rho=cfg.heston_market.rho, S0=cfg.model.S0, r=cfg.model.r,
)
iv_market = heston_mkt.implied_vol_surface_ASV(strikes, maturities_arr)

market_prices = np.zeros((len(maturities_arr), len(strikes)))
for i, T in enumerate(maturities_arr):
    for j, K in enumerate(strikes):
        market_prices[i, j] = BlackScholes.call_price(cfg.model.S0, K, T, cfg.model.r, iv_market[i, j])

print(f'Market prices (proxy ASV):')
print(f'  shape={market_prices.shape}')
print(f'  range=[{market_prices.min():.4f}, {market_prices.max():.4f}]')

In [ ]:
# Run mini calibration (30 L-BFGS-B iterations)
from volsig.calibration.optimizer import SignatureCalibrator
import time

calibrator = SignatureCalibrator(
    market_prices=market_prices,
    strikes=strikes,
    maturities=maturities_arr,
    pricer=multi_pricer,
    S0=cfg.model.S0,
    r=cfg.model.r,
    sigma0=cfg.heston_market.sigma0,
    weight_scheme=cfg.calibration.weight_scheme,
    box_bounds=tuple(cfg.calibration.box_bounds),
    max_iter=cfg.calibration.max_iter,
    tol=cfg.calibration.tolerance,
)

print(f'Starting mini calibration (max_iter={cfg.calibration.max_iter})...')
t0 = time.time()
result = calibrator.calibrate(l0=None)
elapsed = time.time() - t0

l_star = result.x
print(f'\n✓ Calibration complete in {elapsed:.1f}s')
print(f'  Final loss L(ℓ*): {result.fun:.4e}')
print(f'  |ℓ*[0]| ≈ {abs(l_star[0]):.4f}  (should be ≈ σ₀={cfg.heston_market.sigma0})')
print(f'  Optimizer success: {result.success}')

In [ ]:
# Compute IV surface from calibrated ℓ*
iv_sig = multi_pricer.implied_vol_surface(l_star)

print('Implied Volatility: Market vs SIG')
print(f'{"T":>5}  {"K":>5}  {"IV_mkt":>8}  {"IV_sig":>8}  {"error":>8}')
for i, T in enumerate(maturities_arr):
    for j, K in enumerate(strikes):
        err = abs(iv_sig[i,j] - iv_market[i,j])
        print(f'{T:>5.2f}  {K:>5.0f}  {iv_market[i,j]:>8.5f}  {iv_sig[i,j]:>8.5f}  {err:>8.5f}')

mean_err = np.nanmean(np.abs(iv_sig - iv_market))
print(f'\nMean absolute IV error: {mean_err:.5f}')
print(f'(Paper achieves ~1e-4 with nMC=800k and 20 maturities×strikes)')

## Paper Results Reference

Expected results from the paper (for verification after full `train.py` runs):

In [ ]:
paper_results = {
    'heston_uncorrelated': {
        'loss_at_convergence': 1.05e-4,
        'IV_error_range': '5e-5 to 1.3e-3',
        'l_star_l0': 0.201202133,   # ≈ σ₀
        'l_star_l2': 1.08471290,    # strong linear dependence on X_t
    },
    'heston_correlated (rho=-0.5)': {
        'loss_at_convergence': 1.46e-3,
        'IV_error_range': '1e-4 to 1e-3',
    },
    'rough_bergomi (shifted-exp fBM)': {
        'loss_at_convergence': 3.5e-4,
        'IV_error_range': '~1e-4',
        'calibration_time_minutes': '17-19 min (RTX 3080 Ti)',
    },
}

print('Paper reported results (Tables 5.1, 5.2, 6.1):')
for exp, res in paper_results.items():
    print(f'\n  {exp}:')
    for k, v in res.items():
        print(f'    {k}: {v}')

print('\nTo reproduce: python train.py --config configs/config.yaml --experiment heston_uncorr')

## What to Do Next

### Full Experiments
```bash
# Heston uncorrelated (Section 5.1) — ~45 min GPU
python train.py --config configs/config.yaml --experiment heston_uncorr

# Heston correlated (Section 5.2) — ~60 min GPU  
python train.py --config configs/config.yaml --experiment heston_corr

# Rough Bergomi fBM primary (Section 6) — ~17-19 min GPU
python train.py --config configs/config.yaml --experiment rough_bergomi

# Evaluate and plot
python evaluate.py --l_star results/heston_uncorr/l_star.npy \
                   --experiment heston_uncorr --plot
```

### Key Implementation Notes (from SIR)

1. **`T_steps_per_unit: 252`** — ASSUMED (paper says "Euler scheme", omits step count). Confidence: 0.55
2. **`box_bounds: [-10, 10]`** — ASSUMED (paper mentions box constraints, bounds unspecified). Confidence: 0.45  
3. **VIX ATMI** — rough Bergomi calibration uses a proxy instead of true nested VIX simulation. See `data/README_data.md`.

### Stage 6 — Results Comparator
Feed your `results/` directory back to the ArXivist pipeline to compare against
paper-reported numbers with statistical significance testing.